# Diagram meshing: showcasing and benchmarking `synthetmic.utils.mesh_diagram` function

<a href="https://colab.research.google.com/github/synthetic-microstructures/synthetmic/blob/main/examples/notebooks/utils/mesh_diagram.ipynb" target="_blank">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

In this example, we show how to use the `synthetmic.utils.mesh_diagram` function.
We also compare its computation time when `parallel=True` (running the batch of points
in parallel) and when `parallel=False`.

> `parallel=True` is recommended to be used only for large numbers of points and/or grains (seeds),
> where distance computation becomes expensive. Using it on low numbers of points and/or grains
> does not provide any computation speed gain.



First, we install necessary packages.

In [ ]:
!pip install synthetmic numpy pandas tabulate

Next, we import necessary modules and functions from the installed packages.

In [ ]:
import time

from synthetmic.utils import mesh_diagram
import numpy as np
from pandas import DataFrame
from pysdot import PowerDiagram
from pysdot.domain_types import ConvexPolyhedraAssembly
from tabulate import tabulate

We create some functions to provide simple (when we know the ground truth)
and that of high dimension cases of power diagram and points.

In [ ]:
def create_simple_case() -> tuple[PowerDiagram, np.ndarray]:
    seeds = np.array([[0.25, 0.25], [0.75, 0.25], [0.75, 0.75], [0.25, 0.75]])
    domain = ConvexPolyhedraAssembly()
    domain.add_box(np.zeros(seeds.shape[1]), np.ones(seeds.shape[1]))
    weights = np.zeros(seeds.shape[0])
    pd = PowerDiagram(positions=seeds, weights=weights, domain=domain)

    points = np.array([[0.1, 0.1], [0.9, 0.2], [0.7, 0.6], [0.3, 0.8]])

    return pd, points


def create_high_dim_case(
    n_grains: int, n_points: int, dim: int
) -> tuple[PowerDiagram, np.ndarray]:
    domain = ConvexPolyhedraAssembly()
    domain.add_box(np.zeros(dim), np.ones(dim))

    weights = np.zeros(n_grains)
    seeds = np.random.rand(n_grains, dim)
    pd = PowerDiagram(positions=seeds, weights=weights, domain=domain)

    points = np.random.rand(n_points, dim)

    return pd, points

We now create functions to check if the `mesh_diagram` function
returns expected results. In addition, a function that compare
computation time is created.

In [ ]:
def check_correctness() -> None:

    results = {"expected": [0,1,2,3]}

    pd, points = create_simple_case()

    for opt in ("no_parallel", "parallel"):
        
        parallel = True if opt == "parallel" else False
        results[opt] = mesh_diagram(points=points, pd=pd, parallel=parallel, batch_size=1)

    
    for k, v in results.items():
        print(f"* {k}: {v}")

def compare_computation_times(n_grains: int, n_points: int, dim: int, batch_size: int) -> None:

    df = DataFrame(index=("no_parallel", "parallel"))

    pd, points = create_high_dim_case(n_grains=n_grains, n_points=n_points, dim=dim)

    for idx in df.index:

        parallel = True if idx == "parallel" else False

        s = time.perf_counter()
        mesh_diagram(points=points, pd=pd, batch_size=batch_size, parallel=parallel)
        e = time.perf_counter()

        df.loc[idx, "Time elapased (s)"] = e - s

   
    df = df.round(decimals=4)

    print(tabulate(df, headers="keys", tablefmt="simple_grid"))

    return None


Let's run `check_correctness` function.

In [ ]:
check_correctness()

Now we run `compare_computation_times` function for
$10^4$ grains and $10^6$ points. We do this for 2D
and 3D cases.

In [ ]:
for dim in (2, 3):
    print(f"* Running comparison for case {dim}D...")
    
    compare_computation_times(
        n_grains=10_000,
        n_points=1_000_000,
        dim=dim,
        batch_size=100
    )

The same thing but for $10^7$ points:

In [ ]:
for dim in (2, 3):
    print(f"* Running comparison for case {dim}D...")
    
    compare_computation_times(
        n_grains=10_000,
        n_points=10_000_000,
        dim=dim,
        batch_size=100
    )